In [1]:
pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.2 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, glob, math, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import re
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict, Counter
from tqdm import tqdm

for p in (r'C:\Users\Admin\Documents\GitHub\claude_plum',
          '/kaggle/input/datasets/qwert123/hetero-data-updated-diffemb',
          '/kaggle/input/datasets/artmak2/hetero-data-updated-diffemb'):
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

In [3]:
KAGGLE_BASE  = '/kaggle/input/datasets/qwerte123/hetero-data-updated-diffemb'
LOCAL_BASE   = r'C:\Users\Admin\Documents\GitHub\claude_plum\data'
BASE         = KAGGLE_BASE if os.path.exists(KAGGLE_BASE) else LOCAL_BASE

DATA_PATH    = os.path.join(BASE, 'heterodata_object12_updated.pt') 
SEQ_PATH     = os.path.join(BASE, 'sequential_data.txt')
CKPT_DIR_IMP = os.path.join(BASE, 'checkpoints_improved') if os.path.exists(LOCAL_BASE) else BASE

SAVE_DIR = ('/kaggle/working/exp_centroid_semantic'
            if os.path.exists('/kaggle')
            else os.path.join(LOCAL_BASE, 'exp_centroid_semantic'))
os.makedirs(SAVE_DIR, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

df = torch.load(DATA_PATH, weights_only=False, map_location='cpu')
embeds = df['item'].x
n_items = embeds.shape[0]
print(f'items: {n_items}  embed_dim: {embeds.shape[1]}')


device: cuda
items: 12101  embed_dim: 100


In [4]:
def load_sequances(SEQ_PATH, zero_based=1):
    seqs = []
    with open(SEQ_PATH, 'r') as f:
        for line in f:
            seq = line.strip().split()
            if len(seq) < 2:
                continue
            if zero_based == 1:
                items = seq[1:]
                seqs.append([int(i) - 1 for i in items])
            else:
                items = seq[1:]
                seqs.append([int(i) for i in items])
    return seqs

def compute_item_popularity(sequences):
    pop = defaultdict(int)
    for seq in sequences:
        for item in seq:
            pop[item] += 1 
    return pop
sequences = load_sequances(SEQ_PATH, zero_based=1)
print(sequences[:5])
popularity = compute_item_popularity(sequences)

[[0, 1, 2, 3, 4], [5, 6, 7, 8, 9, 3, 10], [3, 11, 12, 13, 14, 15, 16, 17, 18], [19, 20, 21, 22, 3, 23], [3, 24, 25, 26, 27, 28, 29, 30, 31]]


In [5]:
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim):
        super().__init__()
        layers, dims = [], [input_dim] + list(hidden_dims) + [output_dim]
        for i, (a, b) in enumerate(zip(dims[:-1], dims[1:])):
            layers.append(nn.Linear(a, b, bias=True))
            if i < len(dims) - 2:
                layers.append(nn.LayerNorm(b))
                layers.append(nn.ReLU())
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

class EMACodebook(nn.Module):
    def __init__(self, codebook_size, emb_dim, beta=0.25, ema_decay=0.99, epsilon=1e-4):
        super().__init__()
        self.codebook_size = codebook_size; self.beta = beta
        self.ema_decay = ema_decay; self.epsilon = epsilon
        emb = F.normalize(torch.randn(codebook_size, emb_dim), p=2, dim=1)
        self.register_buffer('emb', emb)
        self.register_buffer('ema_count', torch.ones(codebook_size))
        self.register_buffer('ema_weight', emb.clone())
        self.register_buffer('initialized', torch.zeros(1, dtype=torch.bool))
    def forward(self, x):
        x_n = F.normalize(x, p=2, dim=1)
        code_n = F.normalize(self.emb, p=2, dim=1)
        ids = (1.0 - x_n @ code_n.T).argmin(dim=1)
        emb = self.emb[ids]
        return self.beta * F.mse_loss(x, emb.detach()), x + (emb - x).detach(), ids
    
class _RQVAEBase(nn.Module):
    def __init__(self, inp_size, hidden_sizes, embed_dim, n_layers,
                 codebook_size=256, beta=0.25, gamma=0.1, ema_decay=0.99, **kwargs):
        super().__init__()
        self.n_layers = n_layers
        self.enc = Encoder(inp_size, hidden_sizes, embed_dim)
        self.dec = Encoder(embed_dim, hidden_sizes[::-1], inp_size)
        self.codebooks = nn.ModuleList([
            EMACodebook(codebook_size, embed_dim, beta=beta, ema_decay=ema_decay)
            for _ in range(n_layers)
        ])
    def forward(self, x):
        x_n = F.normalize(x, p=2, dim=1)
        r, sids = self.enc(x_n), []
        for cb in self.codebooks:
            _, emb_st, ids = cb(r)
            r = r - emb_st.detach()
            sids.append(ids)
        return {'sids': sids}
    
class RQVAE_Improved(_RQVAEBase):
    def __init__(self, *args, temperature=0.07, **kwargs):
        super().__init__(*args, **kwargs); self.temperature = temperature

def infer_hidden_sizes(state_dict):
    items = [(k, v) for k, v in state_dict.items()
             if k.startswith('enc.net.') and k.endswith('.weight') and v.dim() == 2]
    items.sort(key=lambda kv: int(re.search(r'enc\.net\.(\d+)\.weight', kv[0]).group(1)))
    return [v.shape[0] for _, v in items][:-1]

def load_rqvae(ckpt_path, model_class, inp_size, device):
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    hp = ckpt['hparams']
    hs = infer_hidden_sizes(ckpt['model_state'])
    kw = {k: hp[k] for k in ('embed_dim','n_layers','codebook_size','beta','gamma','ema_decay') if k in hp}
    if 'temperature' in hp: kw['temperature'] = hp['temperature']
    model = model_class(inp_size=inp_size, hidden_sizes=hs, **kw).to(device)
    model.load_state_dict(ckpt['model_state'])
    model.eval()
    return model, hp

def checkpoint_input_dim(path):
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    return int(ckpt['model_state']['enc.net.0.weight'].shape[1])

imp_paths = glob.glob(os.path.join(CKPT_DIR_IMP, 'rqvae_improved_s*.pt'))

target_dim = int(embeds.shape[1])
compatible_imp_paths = []
for p in sorted(imp_paths):
    try:
        if checkpoint_input_dim(p) == target_dim:
            compatible_imp_paths.append(p)
    except Exception as e:
        print(f'Skipping {os.path.basename(p)}: {e}')

if not compatible_imp_paths:
    seen = {os.path.basename(p): checkpoint_input_dim(p) for p in imp_paths}
    raise FileNotFoundError(
        f'No rqvae_improved_s*.pt checkpoint with input_dim={target_dim}. '
        f'Found input dims: {seen}'
    )

best_imp_path = sorted(compatible_imp_paths)[-1]

rqvae, hp = load_rqvae(best_imp_path, RQVAE_Improved, embeds.shape[1], device)
print('Using', os.path.basename(best_imp_path), 'n_layers =', hp['n_layers'], 'codebook_size =', hp['codebook_size'])

Using rqvae_improved_s5.pt n_layers = 4 codebook_size = 256


In [6]:
@torch.no_grad
def encode_base_sids(rqvae, embeds, device, batch_size=1024):
    rqvae.eval()
    n_embeds = len(embeds)
    sids = [None]*n_embeds
    for i in range(0, n_embeds, batch_size):
        end = min(n_embeds, i + batch_size)
        sids_cur = rqvae(embeds[i:end].to(device))['sids']
        codes = torch.stack(sids_cur, dim=1).cpu().tolist() 
        for i, c in zip(range(i, end), codes):
            sids[i] = tuple(c)
    return sids

@torch.no_grad
def get_encodings(rqvae, embeds, device, batch_size=1024):
    rqvae.eval()
    n_embeds = len(embeds)
    encodings = []
    res = [] 
    for i in range(0, n_embeds, batch_size):
        end = min(n_embeds, i+batch_size)
        x = F.normalize(embeds[i:end].to(device), p=2, dim=1)
        encodings_cur = rqvae.enc(x)
        resudials = encodings_cur.clone()
        for codebook in rqvae.codebooks:
            _, emb_st, _ = codebook(resudials)
            resudials = resudials - emb_st.detach()
        encodings.append(encodings_cur)
        res.append(resudials)
    return torch.cat(encodings, 0), torch.cat(res, 0)

def build_collision_mask(base):
    counts = Counter(base)
    return np.array([counts[s] > 1 for s in base], dtype=bool)

def kmeans_residuals_codes(residuals, k, collision_mask, n_iter=30, seed=0):
    rng = np.random.RandomState(seed)
    R = residuals.cpu().numpy().astype(np.float64)
    R = R/(np.linalg.norm(R, axis=1, keepdims=True) + 1e-12)
    coll_idx = np.where(collision_mask)[0]
    if len(coll_idx) == 0:
        return [0] * len(residuals)
    X = R[coll_idx]
    init = X[rng.choice(len(X), size=min(k, len(X)), replace=False)]
    C = init.copy()
    for _ in range(n_iter):
        d = 1.0 - X@C.T
        a = d.argmin(axis=1)
        new_C = np.zeros_like(C)
        for j in range(len(C)):
            mask = (a==j)   
            if mask.any():
                cur_c = X[mask].mean(axis=0) 
                new_C[j] = cur_c / (np.linalg.norm(cur_c) + 1e-12)
            else:
                new_C[j] = C[j]
        if np.allclose(new_C, C, atol=1e-6):
            C = new_C 
            break
        C = new_C
    d = 1.0 - X@C.T
    a = d.argmin(axis=1)
    codes = [0] * len(residuals)
    for idx, c in zip(coll_idx, a):
        codes[int(idx)] = int(c)
    return codes

In [7]:
base_sids = encode_base_sids(rqvae, embeds, device)
print(base_sids[:5])
encodings, residuals = get_encodings(rqvae, embeds, device)
print(f"Encodings shape: {encodings.shape},\nResiduals after RQ-VAE shape: {residuals.shape}")

coll_mask = build_collision_mask(base_sids)
print(f'items in any collision cluster: {coll_mask.sum()} / {len(base_sids)}')

[(223, 152, 96, 100), (223, 12, 176, 40), (166, 106, 243, 196), (194, 100, 190, 50), (81, 250, 102, 96)]
Encodings shape: torch.Size([12101, 32]),
Residuals after RQ-VAE shape: torch.Size([12101, 32])
items in any collision cluster: 617 / 12101


In [8]:
max_dupe_obs = max(Counter(base_sids).values())
K_SEMANTIC = max(8, max_dupe_obs)
print(f'max observed cluster size = {max_dupe_obs}  ->  K_SEMANTIC = {K_SEMANTIC}')

semantic_codes = kmeans_residuals_codes(residuals, K_SEMANTIC, coll_mask, n_iter=25, seed=42)
print(f'semantic_codes: {len(semantic_codes)}  unique used codes: {len(set(semantic_codes))}')

max observed cluster size = 6  ->  K_SEMANTIC = 8
semantic_codes: 12101  unique used codes: 8


In [9]:
encodings

tensor([[ 8.4994e-04,  9.7791e-03, -3.5344e-02,  ...,  2.2830e-02,
         -3.5021e-02,  3.2136e-02],
        [ 2.1829e-02, -7.7211e-02,  7.6869e-03,  ..., -3.4974e-02,
         -2.3439e-02,  4.6545e-02],
        [-1.4044e-04, -2.7152e-01, -2.0079e-02,  ...,  1.2157e-01,
         -1.4320e-01,  6.8504e-02],
        ...,
        [-1.0664e-01,  1.1416e-01, -6.9682e-02,  ...,  9.3598e-02,
          8.3275e-02, -6.2421e-02],
        [-8.6076e-02,  1.1406e-01, -4.5972e-02,  ...,  1.2161e-01,
          6.2148e-02, -1.1030e-01],
        [-6.7992e-02,  1.4315e-01, -2.4045e-02,  ...,  8.7943e-02,
          5.8661e-02, -9.3605e-02]], device='cuda:0')

In [10]:
def assign_sids(base_sids, popularity=None, encodings=None, semantic_codes=None, tiebreak='count', semantic_tiebreak_strict = False):
    res_sids = {}
    clusters = defaultdict(list)
    for ind, sid in enumerate(base_sids):
        clusters[sid].append(ind)
        
    if tiebreak == 'popularity':
        for cluster_sid, inds in clusters.items():
            inds.sort(key = lambda x: -popularity[x])
    
    elif tiebreak == 'centroid':
        encods = F.normalize(encodings, p=2, dim=1).detach().cpu().numpy()
        for cluster_sid, inds in clusters.items():
            if len(inds) == 1:
                continue
            sub = encods[inds]
            sub_mean = sub.mean(axis=0)
            sub_mean = sub_mean/(np.linalg.norm(sub_mean)+1e-12)
            d = 1.0 - sub@sub_mean
            a = np.argsort(d)
            clusters[cluster_sid] = [inds[j] for j in a]


    elif tiebreak == 'semantic':
         if semantic_tiebreak_strict:
             used = defaultdict(set)
         for cluster_sid, inds in clusters.items():
            for ind in inds:
                if semantic_tiebreak_strict == False:
                    res_sids[ind] = cluster_sid + (int(semantic_codes[ind]),)
                else:
                    code = int(semantic_codes[ind])
                    while code in used[cluster_sid]:
                        code+=1
                    used[cluster_sid].add(code)
                    res_sids[ind] = cluster_sid + (code,)

    if tiebreak != 'semantic':      
        for cluster_sid, inds in clusters.items():
                for cnt, ind in enumerate(inds):
                    res_sids[ind] = cluster_sid + (cnt,)

    sid_to_item = defaultdict(list)
    for item, sid in res_sids.items():
        sid_to_item[sid].append(item)
    if tiebreak == 'semantic':
        max_dupe = 1 + max(sid[-1] for sid in res_sids.values())
    else:
        max_dupe = max(len(ids) for ids in clusters.values())
    return res_sids, sid_to_item, max_dupe


In [11]:
def build_trie(sid_to_item) -> dict:
    trie = {}
    for sid, ids in sid_to_item.items():
        node = trie
        for c in sid[:-1]:
            node = node.setdefault(c, {})
        node[sid[-1]] = ids[0]
    return trie

PAD_ID, BOS_ID = 0, 1
MAX_HIST_LEN = 20
D_MODEL, N_HEADS, N_LAYERS_GPT, DROPOUT = 256, 8, 4, 0.1
BATCH_SIZE, LR, WARMUP_STEPS = 256, 1e-3, 500
N_EPOCHS = 30
BEAM_SIZE = 32
EVAL_KS = [1, 5, 10, 20]

def build_pipeline(tiebreak='count', semantic_tiebreak_strict=False):
    L, K = hp['n_layers'], hp['codebook_size']

    if tiebreak == 'base_only':
        full = {i: tuple(sid) for i, sid in enumerate(base_sids)}
        sid2item = defaultdict(list)
        for item, sid in full.items():
            sid2item[sid].append(item)
        max_dupe = 0
        n_levels = L
        lev_off = [2 + l * K for l in range(L)]
        vocab = 2 + L * K

    else:
        if tiebreak == 'popularity':
            full, sid2item, max_dupe = assign_sids(
                base_sids=base_sids,
                tiebreak='popularity',
                popularity=popularity,
            )

        elif tiebreak == 'centroid':
            full, sid2item, max_dupe = assign_sids(
                base_sids=base_sids,
                tiebreak='centroid',
                encodings=encodings,
            )

        elif tiebreak == 'semantic':
            full, sid2item, max_dupe = assign_sids(
                base_sids=base_sids,
                tiebreak='semantic',
                semantic_codes=semantic_codes,
                semantic_tiebreak_strict=semantic_tiebreak_strict,
            )

        else:
            full, sid2item, max_dupe = assign_sids(
                base_sids=base_sids,
                tiebreak='count',
            )

        n_levels = L + 1
        lev_off = [2 + l * K for l in range(L)] + [2 + L * K]
        vocab = 2 + L * K + max_dupe

    def item_to_tokens(i):
        sid = full[i]
        return [sid[l] + lev_off[l] for l in range(n_levels)]

    def history_to_tokens(ids):
        toks = [BOS_ID]
        for iid in ids:
            toks.extend(item_to_tokens(int(iid)))
        return toks

    trie = build_trie(sid2item)

    return dict(
        full=full,
        sid2item=sid2item,
        max_dupe=max_dupe,
        n_levels=n_levels,
        lev_off=lev_off,
        vocab=vocab,
        item_to_tokens=item_to_tokens,
        history_to_tokens=history_to_tokens,
        trie=trie,
    )


In [12]:
pipes = {
    'base_only': build_pipeline('base_only'),
    'count': build_pipeline('count'),
    'centroid': build_pipeline('centroid'),
    'popularity': build_pipeline('popularity'),

    'semantic_soft': build_pipeline(
        tiebreak='semantic',
        semantic_tiebreak_strict=False,
    ),

    'semantic_strict': build_pipeline(
        tiebreak='semantic',
        semantic_tiebreak_strict=True,
    ),
}

for tb, p in pipes.items():
    n_items = len(p["full"])
    n_unique_sids = len(p["sid2item"])
    n_collapsed = n_items - n_unique_sids

    print(
        f'{tb:15s}  '
        f'vocab={p["vocab"]:5d}  '
        f'max_dupe={p["max_dupe"]:3d}  '
        f'unique_sids={n_unique_sids:6d}  '
        f'collapsed={n_collapsed:6d}'
    )


base_only        vocab= 1026  max_dupe=  0  unique_sids= 11754  collapsed=   347
count            vocab= 1032  max_dupe=  6  unique_sids= 12101  collapsed=     0
centroid         vocab= 1032  max_dupe=  6  unique_sids= 12101  collapsed=     0
popularity       vocab= 1032  max_dupe=  6  unique_sids= 12101  collapsed=     0
semantic_soft    vocab= 1034  max_dupe=  8  unique_sids= 11849  collapsed=   252
semantic_strict  vocab= 1037  max_dupe= 11  unique_sids= 12101  collapsed=     0


In [13]:
def cnt0_set(pipe):
    return {i for i, sid in pipe['full'].items() if sid[-1] == 0}

sets = {
    'count': cnt0_set(pipes['count']),
    'centroid': cnt0_set(pipes['centroid']),
    'popularity': cnt0_set(pipes['popularity']),
    'semantic_soft': cnt0_set(pipes['semantic_soft']),
    'semantic_strict': cnt0_set(pipes['semantic_strict']),
}


def overlap(a, b):
    return len(sets[a] & sets[b])

def union_size(a, b):
    return len(sets[a] | sets[b])

def jaccard(a, b):
    u = union_size(a, b)
    return overlap(a, b) / u if u > 0 else 0.0

print("cnt=0 set sizes")
for name, s in sets.items():
    print(f"|cnt=0 {name:10s}| = {len(s)}")

print("\npairwise overlaps")
names = list(sets.keys())

for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = names[i], names[j]
        print(
            f"{a:10s} ∩ {b:10s}: "
            f"{overlap(a, b):6d}  "
            f"Jaccard={jaccard(a, b):.4f}"
        )

print("\ntriple overlaps")
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        for k in range(j + 1, len(names)):
            a, b, c = names[i], names[j], names[k]
            inter = sets[a] & sets[b] & sets[c]
            print(f"{a:10s} ∩ {b:10s} ∩ {c:10s}: {len(inter):6d}")

all_inter = set.intersection(*sets.values())
all_union = set.union(*sets.values())

print("\nall strategies")
print(f"all intersection: {len(all_inter):6d}")
print(f"all union:        {len(all_union):6d}")
print(f"all Jaccard:      {len(all_inter) / len(all_union) if len(all_union) > 0 else 0.0:.4f}")


cnt=0 set sizes
|cnt=0 count     | = 11754
|cnt=0 centroid  | = 11754
|cnt=0 popularity| = 11754
|cnt=0 semantic_soft| = 11555
|cnt=0 semantic_strict| = 11528

pairwise overlaps
count      ∩ centroid  :  11653  Jaccard=0.9830
count      ∩ popularity:  11659  Jaccard=0.9840
count      ∩ semantic_soft:  11513  Jaccard=0.9760
count      ∩ semantic_strict:  11513  Jaccard=0.9782
centroid   ∩ popularity:  11625  Jaccard=0.9783
centroid   ∩ semantic_soft:  11515  Jaccard=0.9763
centroid   ∩ semantic_strict:  11509  Jaccard=0.9776
popularity ∩ semantic_soft:  11516  Jaccard=0.9765
popularity ∩ semantic_strict:  11511  Jaccard=0.9779
semantic_soft ∩ semantic_strict:  11528  Jaccard=0.9977

triple overlaps
count      ∩ centroid   ∩ popularity:  11598
count      ∩ centroid   ∩ semantic_soft:  11506
count      ∩ centroid   ∩ semantic_strict:  11506
count      ∩ popularity ∩ semantic_soft:  11507
count      ∩ popularity ∩ semantic_strict:  11507
count      ∩ semantic_soft ∩ semantic_strict:  11513

In [14]:
class GPT2Rec(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, max_seq_len, n_levels, dropout=0.1):
        super().__init__()
        self.d_model, self.n_levels, self.max_seq_len = d_model, n_levels, max_seq_len
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.lvl_emb = nn.Embedding(n_levels, d_model)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=4*d_model,
                                         dropout=dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)
                if m.padding_idx is not None: m.weight.data[m.padding_idx].zero_()
    def _level_ids(self, T, dev):
        ids = torch.zeros(T, dtype=torch.long, device=dev)
        for pos in range(1, T): ids[pos] = (pos - 1) % self.n_levels
        return ids.unsqueeze(0)
    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0)
        lvl = self._level_ids(T, x.device).expand(B, -1)
        h = self.tok_emb(x) + self.pos_emb(pos) + self.lvl_emb(lvl)
        causal = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
        pad_m = (x == PAD_ID)
        for layer in self.transformer.layers:
            h = layer(h, src_mask=causal, src_key_padding_mask=pad_m)
            h = h.masked_fill(pad_m.unsqueeze(-1), 0.0)
        if self.transformer.norm is not None: h = self.transformer.norm(h)
        return self.lm_head(self.ln_f(h))

class RecDataset(Dataset):
    def __init__(self, samples, max_hist_len, n_levels, full_supervision, item_to_tokens, history_to_tokens):
        self.samples=samples; self.max_hist_len=max_hist_len; self.n_levels=n_levels
        self.full_supervision=full_supervision
        self.item_to_tokens=item_to_tokens; self.history_to_tokens=history_to_tokens
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        ctx, tgt = self.samples[idx]
        ctx = ctx[-self.max_hist_len:]
        inp = self.history_to_tokens(ctx) + self.item_to_tokens(tgt)
        inp = torch.tensor(inp, dtype=torch.long)
        lbl = inp[1:].clone()
        if not self.full_supervision: lbl[:-(self.n_levels + 1)] = -100
        lbl = torch.cat([lbl, torch.tensor([-100])])
        return inp, lbl

def collate_fn(batch):
    inps, lbls = zip(*batch)
    M = max(x.shape[0] for x in inps)
    pi, pl = [], []
    for inp, lbl in zip(inps, lbls):
        pad = M - inp.shape[0]
        pi.append(F.pad(inp, (pad, 0), value=PAD_ID))
        pl.append(F.pad(lbl, (pad, 0), value=-100))
    return torch.stack(pi), torch.stack(pl)

@torch.no_grad()
def beam_search(model, ctx_tok, trie, beam_size, device, level_offsets):
    model.eval()
    n_levels = len(level_offsets); ctx = ctx_tok.to(device)
    logits0 = model(ctx.unsqueeze(0))[0, -1, :]
    beams = [(logits0[c + level_offsets[0]].item(), (c,), sub) for c, sub in trie.items()]
    beams.sort(key=lambda x: -x[0]); beams = beams[:beam_size]
    for lvl in range(1, n_levels):
        code_toks = torch.tensor(
            [[codes[l] + level_offsets[l] for l in range(len(codes))] for _, codes, _ in beams],
            device=device)
        batch = torch.cat([ctx.unsqueeze(0).expand(len(beams), -1), code_toks], dim=1)
        logits = model(batch)[:, -1, :]
        new_beams, is_last = [], (lvl == n_levels - 1)
        for i, (score, codes, node) in enumerate(beams):
            for c, child in node.items():
                ns = score + logits[i, c + level_offsets[lvl]].item()
                new_beams.append((ns, child) if is_last else (ns, codes + (c,), child))
        new_beams.sort(key=lambda x: -x[0])
        if is_last: return new_beams
        beams = new_beams[:beam_size]
    return []

def evaluate(samples, model, trie, level_offsets, history_to_tokens, beam_size, ks, device, max_hist_len):
    hits, ndcg, total = defaultdict(int), defaultdict(float), 0
    for ctx, tgt in tqdm(samples, leave=False):
        ctx_tok = torch.tensor(history_to_tokens(ctx[-max_hist_len:]), dtype=torch.long)
        ranked = beam_search(model, ctx_tok, trie, beam_size, device, level_offsets)
        ranked_ids = [iid for _, iid in ranked]
        for k in ks:
            top_k = ranked_ids[:k]
            if tgt in top_k:
                hits[k] += 1; ndcg[k] += 1.0 / math.log2(top_k.index(tgt) + 2)
        total += 1
    return {**{f'Recall@{k}': round(hits[k]/total, 4) for k in ks},
            **{f'NDCG@{k}':   round(ndcg[k]/total, 4) for k in ks}}

In [15]:
hist = df['user', 'rated', 'item'].history
def make_split(split_key, padded=False):
    item_ids = hist[split_key]['item_ID']; item_next = hist[split_key]['item_ID_next']
    out = []
    for u in range(len(item_next)):
        ctx = [int(x) for x in (item_ids[u].tolist() if padded else item_ids[u]) if int(x) >= 0]
        tgt = int(item_next[u])
        if not ctx or tgt < 0 or tgt >= n_items: continue
        out.append((ctx, tgt))
    return out

samples_train = make_split('train', padded=False)
samples_val   = make_split('valid', padded=True)
samples_test  = make_split('test',  padded=True)
print(f'train={len(samples_train)}  val={len(samples_val)}  test={len(samples_test)}')

train=22363  val=22363  test=22363


In [16]:
def set_all_seeds(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def train_and_eval(pipe, n_epochs=N_EPOCHS, seed=0):
    set_all_seeds(seed)

    max_seq = 1 + (MAX_HIST_LEN + 1) * pipe['n_levels']
    ds_tr = RecDataset(samples_train, MAX_HIST_LEN, pipe['n_levels'], True,
                       pipe['item_to_tokens'], pipe['history_to_tokens'])
    ds_va = RecDataset(samples_val,   MAX_HIST_LEN, pipe['n_levels'], False,
                       pipe['item_to_tokens'], pipe['history_to_tokens'])

    g = torch.Generator()
    g.manual_seed(seed)
    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                       collate_fn=collate_fn, generator=g)
    dl_va = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False,
                       collate_fn=collate_fn)

    model = GPT2Rec(pipe['vocab'], D_MODEL, N_HEADS, N_LAYERS_GPT,
                    max_seq, pipe['n_levels'], DROPOUT).to(device)
    opt = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total = n_epochs * len(dl_tr)
    sch = optim.lr_scheduler.LambdaLR(opt, lambda s: (
        s / max(1, WARMUP_STEPS) if s < WARMUP_STEPS
        else max(0.05, 0.5 * (1.0 + math.cos(math.pi * (s - WARMUP_STEPS) / max(1, total - WARMUP_STEPS))))))

    best_val = float('inf')
    best_state = None

    for epoch in range(1, n_epochs + 1):
        model.train()
        tr_loss, n = 0.0, 0

        for inp, lbl in dl_tr:
            inp, lbl = inp.to(device), lbl.to(device)
            loss = F.cross_entropy(
                model(inp).view(-1, pipe['vocab']),
                lbl.view(-1),
                ignore_index=-100,
            )
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sch.step()
            tr_loss += loss.item()
            n += 1

        model.eval()
        va, m = 0.0, 0
        with torch.no_grad():
            for inp, lbl in dl_va:
                inp, lbl = inp.to(device), lbl.to(device)
                va += F.cross_entropy(
                    model(inp).view(-1, pipe['vocab']),
                    lbl.view(-1),
                    ignore_index=-100,
                ).item()
                m += 1

        va /= max(1, m)
        if va < best_val:
            best_val = va
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if epoch == 1 or epoch % 5 == 0 or epoch == n_epochs:
            print(f'  ep {epoch:3d}  train_loss={tr_loss/max(1,n):.4f}  val_loss={va:.4f}  best={best_val:.4f}')

    model.load_state_dict(best_state)
    res = evaluate(samples_test, model, pipe['trie'], pipe['lev_off'],
                   pipe['history_to_tokens'], BEAM_SIZE, EVAL_KS, device, MAX_HIST_LEN)
    res['val_loss_best'] = round(best_val, 4)

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return res

In [17]:
TRAIN_SEEDS = [3, 4]
RUN_TIEBREAKS = ['base_only', 'count', 'centroid', 'popularity', 'semantic_strict']

rows = []
out_path = os.path.join(SAVE_DIR, 'tiebreak_repeated_train_seeds.csv')
summary_path = os.path.join(SAVE_DIR, 'tiebreak_repeated_train_seeds_summary.csv')
delta_path = os.path.join(SAVE_DIR, 'tiebreak_repeated_train_seeds_deltas_vs_count.csv')

for train_seed in TRAIN_SEEDS:
    print(f'\n################ train_seed = {train_seed} ################')

    for tb in RUN_TIEBREAKS:
        print(f'\n=== train_seed={train_seed}  tie_break={tb} ===')
        metrics = train_and_eval(pipes[tb], seed=train_seed)

        row = dict(
            train_seed=int(train_seed),
            tiebreak=tb,
            vocab=pipes[tb]['vocab'],
            max_dupe=pipes[tb]['max_dupe'],
            unique_sids=len(pipes[tb]['sid2item']),
            collapsed=len(pipes[tb]['full']) - len(pipes[tb]['sid2item']),
            **metrics,
        )
        rows.append(row)
        print(row)

        pd.DataFrame(rows).to_csv(out_path, index=False)

res_df = pd.DataFrame(rows)
print('\n=== Raw results ===')
print(res_df)

metric_cols = [
    c for c in res_df.columns
    if c.startswith('Recall@') or c.startswith('NDCG@') or c == 'val_loss_best'
]

summary = (
    res_df
    .groupby('tiebreak')[metric_cols]
    .agg(['mean', 'std', 'min', 'max'])
    .round(6)
)
summary.to_csv(summary_path)

print('\n=== Summary by tiebreak ===')
print(summary)

wide = res_df.pivot_table(
    index='train_seed',
    columns='tiebreak',
    values=metric_cols,
)

deltas = []
for metric in metric_cols:
    if (metric, 'count') not in wide.columns:
        continue

    for tb in RUN_TIEBREAKS:
        if tb == 'count' or (metric, tb) not in wide.columns:
            continue

        delta = wide[(metric, tb)] - wide[(metric, 'count')]
        deltas.append(dict(
            metric=metric,
            tiebreak=tb,
            mean_delta=delta.mean(),
            std_delta=delta.std(),
            min_delta=delta.min(),
            max_delta=delta.max(),
        ))

delta_df = pd.DataFrame(deltas).round(6)
delta_df.to_csv(delta_path, index=False)

print('\n=== Delta vs count, paired by train_seed ===')
print(delta_df)

print(f'\nSaved raw results to: {out_path}')
print(f'Saved summary to:     {summary_path}')
print(f'Saved deltas to:      {delta_path}')


################ train_seed = 3 ################

=== train_seed=3  tie_break=base_only ===


/tmp/ipykernel_57/2259215166.py:10: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(enc, num_layers=n_layers)
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:823: UserWarning: Support for mismatched src_key_padding_mask and src_mask is deprecated. Use same type for both instead.
  src_key_padding_mask = F._canonical_mask(


  ep   1  train_loss=6.6125  val_loss=5.8460  best=5.8460
  ep   5  train_loss=3.4377  val_loss=2.8141  best=2.8141
  ep  10  train_loss=2.2573  val_loss=1.6644  best=1.6644
  ep  15  train_loss=2.0277  val_loss=1.5080  best=1.5080
  ep  20  train_loss=1.8909  val_loss=1.4040  best=1.4040
  ep  25  train_loss=1.7900  val_loss=1.3354  best=1.3354
  ep  30  train_loss=1.7564  val_loss=1.3172  best=1.3172


{'train_seed': 3, 'tiebreak': 'base_only', 'vocab': 1026, 'max_dupe': 0, 'unique_sids': 11754, 'collapsed': 347, 'Recall@1': 0.007, 'Recall@5': 0.0262, 'Recall@10': 0.0474, 'Recall@20': 0.0838, 'NDCG@1': 0.007, 'NDCG@5': 0.0163, 'NDCG@10': 0.0231, 'NDCG@20': 0.0323, 'val_loss_best': 1.3172}

=== train_seed=3  tie_break=count ===
  ep   1  train_loss=5.9962  val_loss=4.3920  best=4.3920
  ep   5  train_loss=2.8634  val_loss=2.1200  best=2.1200
  ep  10  train_loss=1.8232  val_loss=1.3846  best=1.3846
  ep  15  train_loss=1.6438  val_loss=1.2666  best=1.2666
  ep  20  train_loss=1.5389  val_loss=1.1830  best=1.1830
  ep  25  train_loss=1.4642  val_loss=1.1312  best=1.1312
  ep  30  train_loss=1.4380  val_loss=1.1180  best=1.1180


{'train_seed': 3, 'tiebreak': 'count', 'vocab': 1032, 'max_dupe': 6, 'unique_sids': 12101, 'collapsed': 0, 'Recall@1': 0.0067, 'Recall@5': 0.025, 'Recall@10': 0.0447, 'Recall@20': 0.0807, 'NDCG@1': 0.0067, 'NDCG@5': 0.0159, 'NDCG@10': 0.0222, 'NDCG@20': 0.0312, 'val_loss_best': 1.118}

=== train_seed=3  tie_break=centroid ===
  ep   1  train_loss=6.0001  val_loss=4.4013  best=4.4013
  ep   5  train_loss=2.8642  val_loss=2.1207  best=2.1207
  ep  10  train_loss=1.8235  val_loss=1.3822  best=1.3822
  ep  15  train_loss=1.6440  val_loss=1.2667  best=1.2667
  ep  20  train_loss=1.5392  val_loss=1.1849  best=1.1849
  ep  25  train_loss=1.4639  val_loss=1.1353  best=1.1353
  ep  30  train_loss=1.4382  val_loss=1.1210  best=1.1210


...

{'train_seed': 4, 'tiebreak': 'base_only', 'vocab': 1026, 'max_dupe': 0, 'unique_sids': 11754, 'collapsed': 347, 'Recall@1': 0.0066, 'Recall@5': 0.026, 'Recall@10': 0.0498, 'Recall@20': 0.0879, 'NDCG@1': 0.0066, 'NDCG@5': 0.0161, 'NDCG@10': 0.0237, 'NDCG@20': 0.0332, 'val_loss_best': 1.2956}

=== train_seed=4  tie_break=count ===
  ep   1  train_loss=5.9397  val_loss=4.3251  best=4.3251
  ep   5  train_loss=2.8510  val_loss=2.1084  best=2.1084
  ep  10  train_loss=1.8303  val_loss=1.3800  best=1.3800
  ep  15  train_loss=1.6469  val_loss=1.2526  best=1.2526
  ep  20  train_loss=1.5404  val_loss=1.1733  best=1.1733
  ep  25  train_loss=1.4641  val_loss=1.1174  best=1.1174
  ep  30  train_loss=1.4377  val_loss=1.1002  best=1.1002


{'train_seed': 4, 'tiebreak': 'count', 'vocab': 1032, 'max_dupe': 6, 'unique_sids': 12101, 'collapsed': 0, 'Recall@1': 0.0065, 'Recall@5': 0.0271, 'Recall@10': 0.0485, 'Recall@20': 0.0858, 'NDCG@1': 0.0065, 'NDCG@5': 0.0166, 'NDCG@10': 0.0234, 'NDCG@20': 0.0328, 'val_loss_best': 1.1002}

=== train_seed=4  tie_break=centroid ===
  ep   1  train_loss=5.9443  val_loss=4.3348  best=4.3348
  ep   5  train_loss=2.8519  val_loss=2.1129  best=2.1129
  ep  10  train_loss=1.8285  val_loss=1.3803  best=1.3803
  ep  15  train_loss=1.6480  val_loss=1.2537  best=1.2537
  ep  20  train_loss=1.5399  val_loss=1.1709  best=1.1709
  ep  25  train_loss=1.4641  val_loss=1.1175  best=1.1175
  ep  30  train_loss=1.4376  val_loss=1.0999  best=1.0999


{'train_seed': 4, 'tiebreak': 'centroid', 'vocab': 1032, 'max_dupe': 6, 'unique_sids': 12101, 'collapsed': 0, 'Recall@1': 0.0059, 'Recall@5': 0.0251, 'Recall@10': 0.046, 'Recall@20': 0.0833, 'NDCG@1': 0.0059, 'NDCG@5': 0.0156, 'NDCG@10': 0.0223, 'NDCG@20': 0.0317, 'val_loss_best': 1.0999}

=== train_seed=4  tie_break=popularity ===
  ep   1  train_loss=5.9355  val_loss=4.3175  best=4.3175
  ep   5  train_loss=2.8471  val_loss=2.1037  best=2.1037
  ep  10  train_loss=1.8281  val_loss=1.3807  best=1.3807
  ep  15  train_loss=1.6480  val_loss=1.2616  best=1.2616
  ep  20  train_loss=1.5405  val_loss=1.1760  best=1.1760
  ep  25  train_loss=1.4636  val_loss=1.1205  best=1.1205
  ep  30  train_loss=1.4383  val_loss=1.1028  best=1.1028


{'train_seed': 4, 'tiebreak': 'popularity', 'vocab': 1032, 'max_dupe': 6, 'unique_sids': 12101, 'collapsed': 0, 'Recall@1': 0.0074, 'Recall@5': 0.0262, 'Recall@10': 0.0478, 'Recall@20': 0.0853, 'NDCG@1': 0.0074, 'NDCG@5': 0.0166, 'NDCG@10': 0.0236, 'NDCG@20': 0.033, 'val_loss_best': 1.1028}

=== train_seed=4  tie_break=semantic_strict ===
  ep   1  train_loss=6.0341  val_loss=4.3893  best=4.3893
  ep   5  train_loss=2.8561  val_loss=2.1363  best=2.1363
  ep  10  train_loss=1.8266  val_loss=1.3737  best=1.3737
  ep  15  train_loss=1.6463  val_loss=1.2634  best=1.2634
  ep  20  train_loss=1.5405  val_loss=1.1747  best=1.1747
  ep  25  train_loss=1.4642  val_loss=1.1206  best=1.1206
  ep  30  train_loss=1.4383  val_loss=1.1038  best=1.1038


{'train_seed': 4, 'tiebreak': 'semantic_strict', 'vocab': 1037, 'max_dupe': 11, 'unique_sids': 12101, 'collapsed': 0, 'Recall@1': 0.0059, 'Recall@5': 0.0268, 'Recall@10': 0.047, 'Recall@20': 0.0802, 'NDCG@1': 0.0059, 'NDCG@5': 0.0162, 'NDCG@10': 0.0227, 'NDCG@20': 0.031, 'val_loss_best': 1.1038}

=== Raw results ===
   train_seed         tiebreak  vocab  max_dupe  unique_sids  collapsed  \
0           3        base_only   1026         0        11754        347   
1           3            count   1032         6        12101          0   
2           3         centroid   1032         6        12101          0   
3           3       popularity   1032         6        12101          0   
4           3  semantic_strict   1037        11        12101          0   
5           4        base_only   1026         0        11754        347   
6           4            count   1032         6        12101          0   
7           4         centroid   1032         6        12101          0   
8      

In [18]:
delta_df

,metric,tiebreak,mean_delta,std_delta,min_delta,max_delta
0,Recall@1,base_only,0.00020,0.000141,0.0001,0.0003
1,Recall@1,centroid,-0.00005,0.000778,-0.0006,0.0005
2,Recall@1,popularity,0.00040,0.000707,-0.0001,0.0009
3,Recall@1,semantic_strict,-0.00045,0.000212,-0.0006,-0.0003
4,Recall@5,base_only,0.00005,0.001626,-0.0011,0.0012
5,Recall@5,centroid,-0.00005,0.002758,-0.0020,0.0019
6,Recall@5,popularity,-0.00070,0.000283,-0.0009,-0.0005
7,Recall@5,semantic_strict,0.00070,0.001414,-0.0003,0.0017
8,Recall@10,base_only,0.00200,0.000990,0.0013,0.0027
9,Recall@10,centroid,-0.00030,0.003111,-0.0025,0.0019


In [19]:
res_df

,train_seed,tiebreak,vocab,max_dupe,unique_sids,collapsed,Recall@1,Recall@5,Recall@10,Recall@20,NDCG@1,NDCG@5,NDCG@10,NDCG@20,val_loss_best
0,3,base_only,1026,0,11754,347,0.0070,0.0262,0.0474,0.0838,0.0070,0.0163,0.0231,0.0323,1.3172
1,3,count,1032,6,12101,0,0.0067,0.0250,0.0447,0.0807,0.0067,0.0159,0.0222,0.0312,1.1180
2,3,centroid,1032,6,12101,0,0.0072,0.0269,0.0466,0.0815,0.0072,0.0168,0.0231,0.0319,1.1210
3,3,popularity,1032,6,12101,0,0.0066,0.0245,0.0438,0.0810,0.0066,0.0154,0.0215,0.0308,1.1224
4,3,semantic_strict,1037,11,12101,0,0.0064,0.0267,0.0473,0.0850,0.0064,0.0166,0.0231,0.0326,1.1064
5,4,base_only,1026,0,11754,347,0.0066,0.0260,0.0498,0.0879,0.0066,0.0161,0.0237,0.0332,1.2956
6,4,count,1032,6,12101,0,0.0065,0.0271,0.0485,0.0858,0.0065,0.0166,0.0234,0.0328,1.1002
7,4,centroid,1032,6,12101,0,0.0059,0.0251,0.0460,0.0833,0.0059,0.0156,0.0223,0.0317,1.0999
8,4,popularity,1032,6,12101,0,0.0074,0.0262,0.0478,0.0853,0.0074,0.0166,0.0236,0.0330,1.1028
9,4,semantic_strict,1037,11,12101,0,0.0059,0.0268,0.0470,0.0802,0.0059,0.0162,0.0227,0.0310,1.1038
